# MASLD-Bench · 01 — Prompt Construction (v2)

Builds the LongevityBench-style evaluation/training prompt corpus for MASLD from the two local folders:

- `~/masld-revision` (9.6 GB: discovery cohorts, Kamzolas release, Fujiwara longitudinal block, plasma proteomics, KG, drugs, Tahoe liver shards)
- `~/masld-clinical-transcriptomics-github` (491 MB: locked refinement pipeline, external data)

Recipe replicated: **Zhavoronkov et al., *Cell* 189:5980–5994 (2026)** — LongevityBench / Longevity-LLM.

**v2 additions over v1:** Fujiwara 271-sample longitudinal block (§3), progression tasks from paired biopsies (§3b), drug–gene interaction corpus (§6), MSigDB/FerrDb/QuickGO/scRNA/DGE knowledge tasks (§7), STRING PPI tasks (§7), Tahoe-100M perturbation builder (§8, phase 2).

**Pipeline stage:** DATA → prompt builders → frozen JSONL. This notebook is the *only* stage that touches raw cohort files; run it once, freeze, and everything downstream reads the frozen JSONLs.

## Split discipline (mirrors the paper's leakage audit)

| Domain | Split unit | Test held out |
|---|---|---|
| Bulk liver RNA-seq (Kamzolas) | **cohort** | EPos (n≈157) + Gubra (n≈40) never in training |
| Fujiwara longitudinal block | **patient prefix** (TU/HU/KU/TH families are series-level) | hold out one series (e.g. KU n=56) as internal test |
| Longitudinal paired biopsies | **subject** | ucam_sanyal 58 pairs — evaluation only |
| Plasma proteomics | **subject** | stratified 20% |
| Blood bulk RNA (GSE267032) | **subject** | stratified 20% |
| Drug / pathway / PPI knowledge | **gene family** | HPP-clustering at 30% sequence identity |
| Tahoe perturbation | **cell line** | hold out 2 cell lines entirely |

Pairwise prompts are always constructed **intra-cohort** (the paper's intra-study rule).

## 0 · Setup
Runtime → Change runtime type → **L4 GPU** or CPU. Mount Drive, import libs.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')MASLD_REV = '/content/drive/MyDrive/masld/data/masld-revision'MASLD_GIT = '/content/drive/MyDrive/masld/data/masld-clinical-transcriptomics-github'OUT      = '/content/drive/MyDrive/masld/prompts'   # frozen JSONL output dirimport os, json, gzip, random, hashlib, itertools, reimport numpy as np, pandas as pdos.makedirs(OUT, exist_ok=True)rng = random.Random(20260929)  # one global seed, recorded in every file headerprint('libs ok')

In [ ]:
def sha16(path):    h = hashlib.sha256()    with open(path,'rb') as f:        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)    return h.hexdigest()[:16]def write_jsonl(records, path, meta):    meta = {'n_records': len(records), 'seed': 20260929, **meta}    with gzip.open(path, 'wt') as f:        f.write(json.dumps({'__meta__': meta}) + '\n')        for r in records: f.write(json.dumps(r) + '\n')    print(f"{path}  ->  {len(records):6d} prompts")def fmt_expr(gene, val):    v = float(val)    s = f'{v:.2f}' if v < 100 else f'{v:.1f}' if v < 10000 else f'{v:.0f}'    return f'{gene}={s}'

## 1 · Bring the data up (v2 tiers)
**Tier 1+2 — upload now (~500 MB):** everything in v1 plus the v2 additions:

| Add in v2 | Size |
|---|---|
| `data/expression_matrix.csv` (Fujiwara 213 samples) | 50 MB |
| `data/metadata_with_ferroptosis_scores.csv` (271 rows) | 0.4 MB |
| `data/drug_data/{genes,drugs,interactions,categories}.tsv` | 25 MB |
| `data/msigdb/c2.all.v2023.2.Hs.json` | 6.9 MB |
| `data/quickgo/*.tsv`, `data/ferrdb_*.csv` | 2 MB |
| `data/singlecell/liver_sc_reference*.csv` | 16 MB |
| `results/decisive_test/t1_ranked_logFC.csv`, `t2_fgsea_all.csv` | 1 MB |
| `results/kamzolas/fujiwara_real_paired.csv` + Fujiwara xlsx pair | 5 MB |
| `data/kg/9606.protein.links.v12.0.txt.gz` + `protein.info.txt` | 85 MB |
| `data/kg/kg_additions_drug_target.csv`, `drug_target_merged.csv` | 9 MB |

**Phase 2 (leave on laptop until §8 runs):** Tahoe-100M liver shards (3.5 GB, 43 parquets), Tabula Sapiens h5ad (941 MB), Li 2025 spatial (379 MB).

```bash
# laptop → Drive, once (v2 additions only; v1 set assumed already up):
# rsync -avP ~/masld-revision/data/expression_matrix.csv \
#   ~/masld-revision/data/metadata_with_ferroptosis_scores.csv \
#   ~/masld-revision/data/drug_data /path/to/Drive/masld/data-adjunct/
```

In [ ]:
expected = [    f'{MASLD_REV}/data/kamzolas/metadata.csv',    f'{MASLD_REV}/data/kamzolas/batch_corrected_counts_(dataset+gender).csv',    f'{MASLD_REV}/data/kamzolas/EPoS_dataset/EPoS_counts.tsv.zip',    f'{MASLD_REV}/data/kamzolas/Gubra_dataset/Normalised_Counts_GUBRA.txt',    f'{MASLD_REV}/data/discovery_cohort_349.csv',    f'{MASLD_REV}/data/expression_matrix.csv',                      # v2    f'{MASLD_REV}/data/metadata_with_ferroptosis_scores.csv',       # v2    f'{MASLD_REV}/data/plasma_reference/plasma_proteins.csv',    f'{MASLD_REV}/data/plasma_reference/plasma_sample_metadata.csv',    f'{MASLD_REV}/data/gse267032/metadata.csv',    f'{MASLD_REV}/data/drug_data/interactions.tsv',                 # v2    f'{MASLD_REV}/data/msigdb/c2.all.v2023.2.Hs.json',              # v2    f'{MASLD_REV}/data/kg/9606.protein.links.v12.0.txt.gz',         # v2    f'{MASLD_REV}/results/decisive_test/t1_ranked_logFC.csv',       # v2]for p in expected:    print(('OK  ' if os.path.exists(p) else 'MISS'), p)

## 2 · Task family A — Kamzolas liver bulk RNA-seq (as v1)
SANYAL n=78 + UCAM n=58 train-side; EPos + Gubra stay held out as frozen test. Four formats, rotating verbalizations.

In [ ]:
GENE_SEL = 'topvar'N_SINGLE = 200N_PAIR   = 100VERBALIZATIONS = { 'instr_intro': [   'You are analyzing human liver biopsy transcriptomics from a MASLD study.',   'The following data come from a liver biopsy of a patient in a metabolic dysfunction-associated steatotic liver disease cohort.',   'Below is the expressed gene profile of a human liver sample from a steatotic liver disease study.', ], 'gene_line': [   'Expression (normalised counts): {features}',   'Gene expression measurements: {features}',   'Transcript levels (counts): {features}', ], 'ask_fibrosis_multiclass': [   'Assign the fibrosis stage of this biopsy: F0, F1, F2, F3, or F4.',   'Which fibrosis stage best matches this profile? Answer with one of F0, F1, F2, F3, F4.', ], 'ask_fibrosis_binary': [   'Does this profile indicate advanced fibrosis (F3-F4)? Answer yes or no.',   'Is this biopsy consistent with advanced fibrosis? Answer with yes or no.', ], 'ask_fibrosis_regression': [   'Predict the numeric fibrosis stage (0-4) of this biopsy.',   'Estimate the exact Kleiner fibrosis stage as a number between 0 and 4.', ], 'ask_nas_regression': [   'Predict the NAFLD Activity Score (0-8) of this biopsy.',   'Estimate the exact NAS as a number between 0 and 8.', ],}def pick(dlist, r):    return dlist[r.randrange(len(dlist))]counts = pd.read_csv(f'{MASLD_REV}/data/kamzolas/batch_corrected_counts_(dataset+gender).csv', index_col=0)meta   = pd.read_csv(f'{MASLD_REV}/data/kamzolas/metadata.csv', index_col=0)counts.columns = [c.strip() for c in counts.columns]shared = [s for s in meta.index if s in counts.columns]counts = counts[shared]; meta = meta.loc[shared]meta['Fibrosis'] = pd.to_numeric(meta['Fibrosis'], errors='coerce')meta['NAS']      = pd.to_numeric(meta['NAS'], errors='coerce')print('Kamzolas core:', counts.shape)

In [ ]:
def top_genes(mat, n):    v = mat.var(axis=1)    return v.sort_values(ascending=False).head(n).index.tolist()def render_profile(sample, genes, mat, r):    vals = mat.loc[genes, sample]    feats = ', '.join(fmt_expr(g, vals[g]) for g in genes)    return pick(VERBALIZATIONS['gene_line'], r).format(features=feats)def clinical_block(age=None, sex=None, t2dm=None):    parts = []    if age is not None and pd.notna(age): parts.append(f'age {int(age)}')    if sex is not None and pd.notna(sex): parts.append('female' if str(sex).lower().startswith('f') else 'male')    if t2dm is not None and pd.notna(t2dm): parts.append('type-2 diabetes' if str(t2dm).lower() in ('yes','1') else 'no type-2 diabetes')    return '; '.join(parts)train_ids = [s for s in meta.index if meta.loc[s,'Dataset'] in ('SANYAL','UCAM')]genes_single = top_genes(counts[train_ids], N_SINGLE)genes_pair   = top_genes(counts[train_ids], N_PAIR)records = []for sid in train_ids:    row = meta.loc[sid]    if pd.isna(row['Fibrosis']): continue    base = (f"{pick(VERBALIZATIONS['instr_intro'], rng)}\n"            f"Patient: {clinical_block(row['Age'], row['Sex'], row['T2DM'])}.\n"            f"{render_profile(sid, genes_single, counts, rng)}\n")    records.append(dict(task='MB-1002', split='train', sample=sid,        prompt=base + pick(VERBALIZATIONS['ask_fibrosis_multiclass'], rng), answer=f"F{int(row['Fibrosis'])}"))    records.append(dict(task='MB-1001', split='train', sample=sid,        prompt=base + pick(VERBALIZATIONS['ask_fibrosis_binary'], rng),        answer='yes' if row['Fibrosis']>=3 else 'no'))    records.append(dict(task='MB-1005', split='train', sample=sid,        prompt=base + pick(VERBALIZATIONS['ask_fibrosis_regression'], rng), answer=f"{row['Fibrosis']:.0f}"))    if pd.notna(row['NAS']):        records.append(dict(task='MB-1006', split='train', sample=sid,            prompt=base + pick(VERBALIZATIONS['ask_nas_regression'], rng), answer=f"{row['NAS']:.0f}"))for c in ('SANYAL','UCAM'):    sids = [s for s in train_ids if meta.loc[s,'Dataset']==c]    for a, b in itertools.combinations(sids, 2):        fa, fb = meta.loc[a,'Fibrosis'], meta.loc[b,'Fibrosis']        if pd.isna(fa) or pd.isna(fb) or fa == fb: continue        records.append(dict(task='MB-1003', split='train', cohort=c, sample=f'{a}|{b}',            prompt=(f"{pick(VERBALIZATIONS['instr_intro'], rng)}\n"                    f"Patient A: {clinical_block(meta.loc[a,'Age'], meta.loc[a,'Sex'], meta.loc[a,'T2DM'])}.\n{render_profile(a, genes_pair, counts, rng)}\n\n"                    f"Patient B: {clinical_block(meta.loc[b,'Age'], meta.loc[b,'Sex'], meta.loc[b,'T2DM'])}.\n{render_profile(b, genes_pair, counts, rng)}\n\n"                    "Which patient has the more advanced fibrosis stage? Answer with A or B."),            answer='A' if fa > fb else 'B'))write_jsonl(records, f'{OUT}/MB-10xx_liver_train.jsonl.gz',            meta=dict(builder='kamzolas SANYAL+UCAM', genes_single=N_SINGLE, genes_pair=N_PAIR,                      counts_sha=sha16(f'{MASLD_REV}/data/kamzolas/batch_corrected_counts_(dataset+gender).csv')))

## 3 · v2 — Fujiwara longitudinal block (271 samples)
`expression_matrix.csv` (213 samples, series-prefixed columns) joined to `metadata_with_ferroptosis_scores.csv` (271 rows; age/sex/fibrosis/NAS/risk parsed from GEO characteristics). This more than **doubles** train-side clinical volume.

Split rule: the TU/HU/KU/TH prefixes are different GEO series of the same Fujiwara study family — hold out **KU (n=56)** as an internal test series, train on TU+HU (+TH).

In [ ]:
# ---- load & parse the Fujiwara metadata ----fmeta = pd.read_csv(f'{MASLD_REV}/data/metadata_with_ferroptosis_scores.csv', index_col=0)def parse_char(s, key):    s = str(s)    m = re.search(key + r'\s*:\s*(.+)$', s)    return m.group(1).strip() if m else Nonefuji = pd.DataFrame({    'sample':  fmeta['title'],    'series':  fmeta['title'].str[:2],    'age':     fmeta['characteristics_ch1.1'].map(lambda s: parse_char(s, 'age')),    'sex':     fmeta['characteristics_ch1.2'].map(lambda s: parse_char(s, 'Sex')),    'fibrosis':fmeta['characteristics_ch1.3'].map(lambda s: parse_char(s, 'fibrosis stage')),    'nas':     fmeta['characteristics_ch1.4'].map(lambda s: parse_char(s, 'nafld activity score')),    'risk':    fmeta['characteristics_ch1.5'].map(lambda s: parse_char(s, 'pls-nafld-based risk prediction')),}).dropna(subset=['sample'])fuji['fibrosis'] = pd.to_numeric(fuji['fibrosis'], errors='coerce')fuji['nas']      = pd.to_numeric(fuji['nas'], errors='coerce')fuji['age']      = pd.to_numeric(fuji['age'], errors='coerce')print(fuji.groupby('series').agg(n=('sample','count'), fib_avail=('fibrosis','count')))# ---- join to the expression matrix ----emat = pd.read_csv(f'{MASLD_REV}/data/expression_matrix.csv', index_col=0)emat.columns = [str(c) for c in emat.columns]col2sample = {c: c.split('.', 1)[1] for c in emat.columns if '.' in c}emat = emat.rename(columns=col2sample)fuji = fuji[fuji['sample'].isin(emat.columns)].reset_index(drop=True)print('joined Fujiwara samples:', len(fuji), '| genes:', emat.shape[0])

In [ ]:
# ---- build the four formats over Fujiwara (train = TU+HU+TH, test = KU) ----fuji_train = fuji[fuji['series'].isin(['TU','HU','TH'])]fuji_test  = fuji[fuji['series']=='KU']fgenes_single = top_genes(emat[list(fuji_train['sample'])], N_SINGLE)fgenes_pair   = top_genes(emat[list(fuji_train['sample'])], N_PAIR)def fuji_records(df, split, task_prefix):    out = []    for _, row in df.iterrows():        if pd.isna(row['fibrosis']): continue        base = (f"{pick(VERBALIZATIONS['instr_intro'], rng)}\n"                f"Patient: {clinical_block(row['age'], row['sex'])}.\n"                f"{render_profile(row['sample'], fgenes_single, emat, rng)}\n")        out.append(dict(task=f'{task_prefix}02', split=split, sample=row['sample'],            prompt=base + pick(VERBALIZATIONS['ask_fibrosis_multiclass'], rng), answer=f"F{int(row['fibrosis'])}"))        out.append(dict(task=f'{task_prefix}01', split=split, sample=row['sample'],            prompt=base + pick(VERBALIZATIONS['ask_fibrosis_binary'], rng),            answer='yes' if row['fibrosis']>=3 else 'no'))        out.append(dict(task=f'{task_prefix}05', split=split, sample=row['sample'],            prompt=base + pick(VERBALIZATIONS['ask_fibrosis_regression'], rng), answer=f"{row['fibrosis']:.0f}"))        if pd.notna(row['nas']):            out.append(dict(task=f'{task_prefix}06', split=split, sample=row['sample'],                prompt=base + pick(VERBALIZATIONS['ask_nas_regression'], rng), answer=f"{row['nas']:.0f}"))    return outrecords_fuji = fuji_records(fuji_train, 'train', 'MB-8')write_jsonl(records_fuji, f'{OUT}/MB-8xxx_fujiwara_train.jsonl.gz',            meta=dict(builder='Fujiwara TU+HU+TH train', source='expression_matrix.csv',                      sha=sha16(f'{MASLD_REV}/data/expression_matrix.csv')))# KU internal test built with identical code, split='test', prefix MB-9 — freeze alongside EPos/Gubra.

### 3b · v2 — Progression tasks from paired biopsies
`fujiwara_real_paired.csv` (58 patients, driver/suppressor/axis scores at both biopsies) + `Both biopsies - refined dataset.xlsx` (histology at both timepoints). Predicting **Δfibrosis from baseline molecular scores** is a task family the LongevityBench paper didn't have — this is your novelty claim.

In [ ]:
pair = pd.read_csv(f'{MASLD_REV}/results/kamzolas/fujiwara_real_paired.csv')# Columns include patient, driver_1/2, suppressor_1/2, axis_1/2, poised_*, *_adj_* (baseline=_1, followup=_2),# Histology.fibrosis, Histology.NAS at the paired timepoints.prog = []for _, row in pair.iterrows():    try:        f1 = float(str(row.get('Histology.fibrosis','')).split('_')[0]) if pd.notna(row.get('Histology.fibrosis')) else np.nan    except Exception:        f1 = np.nan    if pd.isna(f1):        continue  # parse histology codes per the refined xlsx if present; skip unparseable rows    prog.append(dict(task='MB-8501', split='train', sample=row['patient'],        prompt=(f"A MASLD patient's baseline liver biopsy shows the following molecular axis scores:\n"                f"driver score {row['driver_1']:.3f}; suppressor score {row['suppressor_1']:.3f}; "                f"axis score {row['axis_1']:.3f}; poised score {row['poised_1']:.3f}.\n"                f"Will the patient's fibrosis stage progress by the follow-up biopsy? Answer yes or no."),        answer='yes'))   # TODO: derive ground truth from Histology.fibrosis at both timepoints in the xlsxwrite_jsonl(prog, f'{OUT}/MB-85xx_progression_train.jsonl.gz',            meta=dict(builder='fujiwara_real_paired', n_pairs=len(pair), note='ground truth needs xlsx join'))print('NOTE: finish the ground-truth join against Both biopsies - refined dataset.xlsx before freezing.')

## 4 · Task family B — plasma proteomics + blood RNA (as v1, stubs)
191-sample plasma reference over Govaere fixed panels (subject split); GSE267032 WBC severity tasks (subject split). Builders identical in shape to §2 — reuse them.

## 5 · Held-out test builders — EPos + Gubra + KU (freeze together)
Same builders, different cohorts, `split='test'`, `MB-2xxx` IDs. KU series from §3 joins this freeze. One manifest, one checksum, forever.

## 6 · v2 — Drug–gene interaction corpus (~250k raw facts)
The DrugBank/DrugCentral/CTD layer becomes retrieval-style **training-only** prompts — your analogue of the paper's UniProt/STRING pretraining family.

In [ ]:
genes = pd.read_csv(f'{MASLD_REV}/data/drug_data/genes.tsv', sep='\t')drugs = pd.read_csv(f'{MASLD_REV}/data/drug_data/drugs.tsv', sep='\t')inter = pd.read_csv(f'{MASLD_REV}/data/drug_data/interactions.tsv', sep='\t')cats  = pd.read_csv(f'{MASLD_REV}/data/drug_data/categories.tsv', sep='\t')print('genes', genes.shape, 'drugs', drugs.shape, 'interactions', inter.shape, 'categories', cats.shape)print('interactions cols:', list(inter.columns))# Generic prompt shape (adapt column names after the print above):#   'Which of the following genes is a known interaction partner of the drug <name>?#    Candidates: A, B, C, D. Answer with one gene symbol.'   (binary/multiclass, train-only)#   'The drug <name> interacts with which gene? Answer with the gene symbol.'# Sample 40-80k from the interaction table; negative candidates drawn from genes.tsv.# Split by gene family (HPP 30% identity clusters) so no gene spans train and any eval use.

In [ ]:
# KG text dump — stage-A continued pretraining (v1 §5 + drug edges)curated = pd.read_csv(f'{MASLD_REV}/data/kamzolas/MASLD_signatures/curated_MASLD_gene_sets.tsv', sep='\t')corpus_lines = [f"{row['ref']}: the gene {row['all_genes']} is associated with {row['early_late_both']} MASLD progression."                for _, row in curated.iterrows()]gwas = pd.read_csv(f'{MASLD_REV}/data/kamzolas/GWAS_catalog/gwas_summary.csv', index_col=0)corpus_lines += [f"GWAS catalog links {row['MAPPED_GENE']} to metabolic traits: {str(row['Trait'])[:180]}."                 for _, row in gwas.head(2000).iterrows()]kg_add = pd.read_csv(f'{MASLD_REV}/results/kg_additions_drug_target.csv')corpus_lines += [f"The drug {row['drug_name']} targets {row['target_gene']} (source: {row['source']})."                 for _, row in kg_add.head(20000).iterrows()]print('corpus lines:', len(corpus_lines))# write OUT/kg_corpus.txt (append STRING §7 + trial outcomes as before)

## 7 · v2 — Knowledge tasks from MSigDB / FerrDb / QuickGO / scRNA / your DGE
Ground truth derived from **data and curated databases**, not free text — the paper's anti-memorization design.

In [ ]:
# (a) MSigDB c2 membership — 7,233 sets, 65k membershipsmsig = json.load(open(f'{MASLD_REV}/data/msigdb/c2.all.v2023.2.Hs.json'))msig_prompts = []for setname, members in list(msig.items())[:4000]:    if not isinstance(members, list) or len(members) < 5: continue    hidden = rng.choice(members)    shown  = [m for m in members if m != hidden][:15]    msig_prompts.append(dict(task='MB-6101', split='train',        prompt=(f'The gene set {setname} contains the following members: {", ".join(shown)}, and others.\n'                f'Which of these additional genes also belongs to {setname}: {hidden}, {rng.choice(msig_prompts and sum([list(v) for v in msig.values() if isinstance(v,list)], [])[:0] or ["TTN","ALB","INS","GAPDH"])}? Answer with one gene symbol.'),        answer=hidden))write_jsonl(msig_prompts, f'{OUT}/MB-61xx_msigdb_train.jsonl.gz', meta=dict(builder='msigdb c2', n_sets=len(msig)))# (b) Masked direction-of-change from YOUR DGE (OpenGenes analogue, real ground truth)t1 = pd.read_csv(f'{MASLD_REV}/results/decisive_test/t1_ranked_logFC.csv')dge_prompts = []for _, row in t1.iterrows():    up = float(row['logFC']) > 0    dge_prompts.append(dict(task='MB-7001', split='train', gene=row['GeneSymbol'],        prompt=('In human liver biopsies of MASLD patients compared to healthy liver tissue, '                'the expression of a gene with the following characteristics changes. '                'Does its expression increase or decrease in MASLD? Answer with increase or decrease.'),        answer='increase' if up else 'decrease'))write_jsonl(dge_prompts, f'{OUT}/MB-70xx_dge_direction_train.jsonl.gz', meta=dict(builder='t1_ranked_logFC', n=len(t1)))# (c) FerrDb ferroptosis roles + QuickGO RCD annotations + scRNA cell-type markers#     shapes: ferrdb_driver.csv / ferrdb_suppressor.csv -> 'Is gene X a driver or suppressor of ferroptosis?'#     quickgo/*.tsv -> 'Is gene X annotated to regulated cell death process <GO term>?'#     singlecell/liver_sc_reference.csv (20 cell types x genes) -> 'Which liver cell type shows highest expression of markers ...?'print('TODO: (c) builders — same pattern, ~10-20k prompts total')

In [ ]:
# (d) STRING PPI tasks — the paper's direct analoguelinks = pd.read_csv(f'{MASLD_REV}/data/kg/9606.protein.links.v12.0.txt.gz', sep=' ')info  = pd.read_csv(f'{MASLD_REV}/data/kg/protein.info.txt', sep='\t')p2g   = dict(zip(info['protein_external_id'], info['preferred_name']))links['g1'] = links['protein1'].map(p2g); links['g2'] = links['protein2'].map(p2g)hi = links[(links['combined_score'] >= 700) & links['g1'].notna() & links['g2'].notna()]print('high-confidence STRING edges:', len(hi))# Prompts: 'Do the proteins encoded by X and Y physically interact (confidence >= 700)? yes/no'#          with negatives sampled from non-edge pairs; sample 20-30k positive + equal negatives.# Split by gene family so no gene appears in both train and eval halves.

## 8 · v2 phase 2 — Tahoe-100M perturbation builder (SynergyAge analogue)
43 liver-duo DE shards already on the laptop (3.5 GB). Each row: drug × cell line × concentration × gene × log2FC × padj.
**Upload the shards to Drive only when you reach this section** (or run this section locally and upload the output JSONL).

In [ ]:
# Run on Colab after uploading shards, or locally with pyarrow installed.import glob, pyarrow.parquet as pqSHARDS = sorted(glob.glob(f'{MASLD_REV}/data/tahoe/pseudobulk_de/*.parquet'))TEST_LINES = ['HCC-2998', 'HCT15']          # held-out cell lines — choose after inspecting coverageTARGET_N = 60_000                            # sample cap; ~10M rows scannedrows, scanned = [], 0rng2 = random.Random(20260930)for shard in SHARDS:    t = pq.read_table(shard, columns=['gene_name','log2FoldChange','padj','drug','Cell_Name_Vevo','concentration'])    df = t.to_pandas()    scanned += len(df)    df = df[(df['padj'] < 0.05) & df['Cell_Name_Vevo'].notna()]    df = df[df['Cell_Name_Vevo'].isin(TEST_LINES) == False]      # train side only here    take = min(len(df), TARGET_N // max(1, len(SHARDS)))    if take: rows.append(df.sample(n=take, random_state=20260930))    if sum(len(r) for r in rows) >= TARGET_N: breaktahoe = pd.concat(rows) if rows else pd.DataFrame()print('scanned rows:', scanned, '| sampled significant DE rows:', len(tahoe))# Perturbation prompts — three formats:#  (i)  direction: 'Under drug D at concentration C in cell line L, does gene G go up or down?' (answer from log2FC sign)#  (ii) magnitude bins: '|log2FC| binned into 0-0.5 / 0.5-1 / 1-2 / >2'#  (iii) pairwise: 'Which gene responds more strongly to drug D in line L: G1 or G2?' (intra cell-line pairs)pert = []for _, r in tahoe.iterrows():    pert.append(dict(task='MB-9101', split='train', cell_line=r['Cell_Name_Vevo'], drug=r['drug'],        prompt=(f"In an in-vitro perturbation experiment, cell line {r['Cell_Name_Vevo']} was treated with "                f"{r['drug']} at {r['concentration']}.\n"                f"Does expression of {r['gene_name']} increase or decrease relative to control? Answer with increase or decrease."),        answer='increase' if r['log2FoldChange'] > 0 else 'decrease'))write_jsonl(pert, f'{OUT}/MB-91xx_tahoe_train.jsonl.gz',            meta=dict(builder='Tahoe-100M liver shards', n_scanned=scanned, test_lines=TEST_LINES))print('NOTE: test-side prompts for the held-out lines built with the same code, split=test -> MB-92xx.')

## 9 · Freeze & manifest (all families)
One manifest, checksums for every JSONL + the kg_corpus.txt. Version bumped on any builder change; all eval runs must then be rebuilt.

In [ ]:
manifest = { 'created': '2026-09-29', 'version': 2, 'seed': 20260929, 'recipe': 'Zhavoronkov et al. Cell 189:5980-5994 (2026), LongevityBench', 'files': {f: sha16(os.path.join(OUT, f)) for f in os.listdir(OUT) if f.endswith(('.jsonl.gz', '.txt'))}}json.dump(manifest, open(f'{OUT}/MANIFEST.json', 'w'), indent=1)print(json.dumps({k: manifest[k] for k in ('version','seed')}, indent=1))print(f"{len(manifest['files'])} frozen files")